# Session 4 recap – Same rules, much smaller model
Runs each group's final system instructions and hardest question on a small model inside this Colab session (nothing leaves it),
next to the answer the large model (via NVIDIA's API) gave in class.
Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes

## 1. Choose the small model
Gemma 3 1B: a Google model about 30 times smaller than the class model (Gemma 4 31B via NVIDIA). Runs fast on a T4 GPU, and even without one.

In [ ]:
MODEL_ID = "google/gemma-3-1b-it"                 # 1 billion parameters, text only (gated: accept Gemma terms)
# MODEL_ID = "google/gemma-3-270m-it"             # 0.27 billion: the extreme (gated)
# MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"         # 1.5 billion, another family, no token needed
# MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"         # 0.5 billion, no token needed
# MODEL_ID = "google/gemma-4-E2B-it"              # Gemma 4, 2.3B effective: same generation as the class model, needs a GPU
# MODEL_ID = "google/medgemma-4b-it"              # medical model, Gemma 3 based (gated), needs a GPU

# Hugging Face token: optional for Gemma 4 (faster downloads), required for gated models.
# Store it once in Colab: key icon (Secrets) -> name HF_TOKEN -> enable notebook access.
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("Using the HF_TOKEN secret")
except Exception:
    if "medgemma" in MODEL_ID or "gemma-3" in MODEL_ID:   # gated models need a token (Gemma 3, MedGemma)
        from getpass import getpass
        os.environ["HF_TOKEN"] = getpass("Hugging Face token: ")

In [ ]:
import torch
from transformers import AutoProcessor, pipeline, BitsAndBytesConfig

# A GPU makes answers much faster: Runtime -> Change runtime type -> T4 GPU
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name}, {gpu.total_memory / 1e9:.0f} GB")
else:
    print("No GPU: fine for models up to ~1.5B (slow); larger models need Runtime -> Change runtime type -> T4 GPU")

QUANTIZE_4BIT = False      # set True if the model still does not fit (smaller, slightly less accurate)
QCONF = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16) if QUANTIZE_4BIT else None
DTYPE = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if torch.cuda.is_available() else torch.float32
MULTIMODAL_GEMMA3 = ("medgemma", "gemma-3-4b", "gemma-3-12b", "gemma-3-27b")   # Gemma 3 1B and 270M are text-only
KIND = "gemma4" if "gemma-4" in MODEL_ID else "multimodal" if any(k in MODEL_ID for k in MULTIMODAL_GEMMA3) else "text"

if KIND == "gemma4":                              # loading as in the Gemma 4 model card
    from transformers import AutoModelForMultimodalLM
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, dtype=DTYPE, device_map="auto", quantization_config=QCONF)
elif KIND == "multimodal":
    pipe = pipeline("image-text-to-text", model=MODEL_ID, dtype=DTYPE, device_map="auto", model_kwargs={"quantization_config": QCONF})
else:
    pipe = pipeline("text-generation", model=MODEL_ID, dtype=DTYPE, device_map="auto", model_kwargs={"quantization_config": QCONF})

def small_ask(question, system="", examples=(), max_new_tokens=500):
    msgs = [{"role": "system", "content": system}] if system else []
    for q, a in examples:
        msgs += [{"role": "user", "content": q}, {"role": "assistant", "content": a}]
    msgs.append({"role": "user", "content": question})
    if KIND == "text":
        return pipe(msgs, max_new_tokens=max_new_tokens, do_sample=False)[0]["generated_text"][-1]["content"]
    msgs = [{"role": m["role"], "content": [{"type": "text", "text": m["content"]}]} for m in msgs]
    if KIND == "multimodal":
        return pipe(text=msgs, max_new_tokens=max_new_tokens, do_sample=False)[0]["generated_text"][-1]["content"]
    inputs = processor.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True,
                                           return_dict=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

print(small_ask("In one sentence: what is an API?"))

## 2. Load the groups' submission files
Upload the `submission_*.json` files (or put them in a Drive folder and set `FOLDER`).

In [ ]:
import glob, json, os
FOLDER = "."                                   # e.g. "/content/drive/MyDrive/session4" after mounting Drive
files = sorted(glob.glob(os.path.join(FOLDER, "submission_*.json")))
if not files:
    from google.colab import files as colab_files
    files = list(colab_files.upload().keys())

subs = [json.load(open(f, encoding="utf-8")) for f in files]
for s in subs:
    print(f"{s['group']:<20} {s['topic']}")

## 3. Hardest question: large model vs small model

In [ ]:
from IPython.display import HTML, display
import html

def side_by_side(title, question, big, small, big_model="large model"):
    cell = "vertical-align:top; padding:8px; border:1px solid #ccc; white-space:pre-wrap; width:50%;"
    display(HTML(f"<h3>{html.escape(title)}</h3><p><b>Question:</b> {html.escape(question)}</p>"
                 f"<table style='width:100%; border-collapse:collapse;'>"
                 f"<tr><th>Large: {html.escape(big_model)} (NVIDIA API)</th><th>Small: {html.escape(MODEL_ID)} (this Colab)</th></tr>"
                 f"<tr><td style='{cell}'>{html.escape(big)}</td><td style='{cell}'>{html.escape(small)}</td></tr></table>"))

for s in subs:
    small = small_ask(s["hardest_question"], s["system"], s["examples"])
    side_by_side(f"{s['group']}: {s['topic']}", s["hardest_question"], s.get("hardest_answer", "(not saved)"), small, s["model"])

## 4. Optional: one group's full test set

In [ ]:
s = subs[0]                                    # pick a group
for r in s["final_answers"]:
    side_by_side(s["group"], r["question"], r["answer"], small_ask(r["question"], s["system"], s["examples"]), s["model"])